In [3]:
#Aprendizado sensível a custo
#Custo em cibersegurança se baseia em:
#a)Falso positivo: é chato bloquear um usuário legítimo, mas o sistema fica seguro
#b)Falso negativo: deixar um hacker passar é catastrófico

In [4]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, roc_auc_score


In [5]:
X_train_np = np.load("../artifacts/data/X_train.npy")
X_test_np  = np.load("../artifacts/data/X_test.npy")
y_train_np = np.load("../artifacts/data/y_train.npy")
y_test_np  = np.load("../artifacts/data/y_test.npy")

print(X_train_np.shape, X_test_np.shape)


(175341, 194) (82332, 194)


In [6]:
X_train_t = torch.tensor(X_train_np, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_np, dtype=torch.float32)

y_train_t = torch.tensor(y_train_np, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_np, dtype=torch.float32)


In [7]:
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=256,
    shuffle=True
)


In [8]:
#aqui preparo o terreno para penalizar o modelo mais fortemente quando ele tem um falso negativo
class CostSensitiveMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze()

#matenho a mesma arquitetura de neurônios, mas pretento ajustar os pesos desses neurônios

In [9]:
#UNSW é um dataset desbalanceado, o modelo comum tende a aprender que é mais
#seguro dizer que quase tudo é tráfego normal para manter acurácia alta
#com o pos-weight, o modelo fica paranóico, preferindo errar por zelo do que deixar ameaça passar
#obs: mantenho os dados originais e íntegros
#resumo: ensino ao modelo que perder um ataque é imperdoável, forço-o a encontrart padrões de intrusão
#mais sutis que o baseline ignoraria. O modelo não deve ignorar um ataque só porque eles aparecem com menos frequência!!!
#não quis inserir dados falsos sejam eles ataques ou tráfego normal.

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CostSensitiveMLP(X_train_np.shape[1]).to(device) #alocação dinâmica: modelo e peso para GPU ou CPU. Todos os elementos envolvidos devem estar na mesma memória

positivos = np.sum(y_train_np == 1)
negativos = np.sum(y_train_np == 0)

#efeito: se houver 10 vezes mais tráfego normal do que ataques, o pos-weight será 10.0
#basicamente digo ao modelo: "cada vez que você deixar passar um ataque, sinta uma cukpa de 10 vezes
pos_weight = torch.tensor([negativos / positivos]).to(device) #uso fórmula estatística para equilibrar a balança

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight) #implementação co cost-sensitive learning

optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Pos weight:", pos_weight.item())


Pos weight: 0.469243596081816


In [10]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/10 - Loss: 0.1118
Epoch 2/10 - Loss: 0.0871
Epoch 3/10 - Loss: 0.0844
Epoch 4/10 - Loss: 0.0828
Epoch 5/10 - Loss: 0.0816
Epoch 6/10 - Loss: 0.0804
Epoch 7/10 - Loss: 0.0794
Epoch 8/10 - Loss: 0.0791
Epoch 9/10 - Loss: 0.0784
Epoch 10/10 - Loss: 0.0781


In [11]:
model.eval()

with torch.no_grad():
    logits = model(X_test_t.to(device))
    probs = torch.sigmoid(logits).cpu().numpy()


In [24]:
#comportamento atual do modelo: tornou-se mais equilibrado
#está separando as classes com mais confiança
#no baseline, o modelo estava tentando apenas diminuir o erro total, oq leva a bloquear todo mundo para subir o recall
#aqui há uma queda de aproximadamente 7% de falsos positivos, com o custo de aproximadamente 1% de recall
#

threshold = 0.49 #alateração no threshold para "tunar" o modelo

y_pred = (probs >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test_np, y_pred).ravel()

recall = tp / (tp + fn)
fpr    = fp / (fp + tn)
auc    = roc_auc_score(y_test_np, probs)

print(f"Recall : {recall:.2%}") #sensibildiade: porcentagem total que o modelo conseguiu pegar
print(f"FPR    : {fpr:.2%}") #alarme falso
print(f"ROC AUC: {auc:.3}") #de 0 a 1, as habilidades do modelo em separar as duas classes


Recall : 95.07%
FPR    : 18.93%
ROC AUC: 0.976
